# Data Understanding and Exploratory Data Analysis

**AI Commerce Analytics Platform**  \n**Purpose:** establish a trusted, business-ready foundation for customer analytics, churn, CLV, delivery, recommendations, sentiment, forecasting, RAG, and executive reporting.

---

## Executive Summary

This notebook validates the ecommerce data model, quantifies data quality, and profiles commercial performance across customers, products, sellers, orders, payments, reviews, and geography. It intentionally separates **diagnostic exploration** from downstream model transformations: source data remains unchanged, while derived features are created in memory with documented assumptions.

The analysis is designed to be rerunnable as data refreshes. All headline metrics are calculated from the loaded source tables rather than hard-coded.

## Business Understanding

The platform needs a reliable view of the end-to-end commerce journey: acquisition and retention, catalog economics, seller operations, fulfilment, payment behavior, and customer sentiment. The central business questions are:

- Which customers, products, sellers, states, and payment methods drive value?
- Where do conversion, fulfilment, delivery experience, or satisfaction show material risk?
- Is the relational source model complete and trustworthy enough for joined analytical and ML feature sets?

## Technical Objectives

1. Load the source CSVs with explicit types and date parsing.
2. Validate expected schemas, keys, duplicates, nulls, and foreign-key relationships.
3. Produce reusable aggregates and interpretable Plotly visuals.
4. Identify feature opportunities, leakage controls, and production data contracts.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

PLOTLY_TEMPLATE = 'plotly_white'
px.defaults.template = PLOTLY_TEMPLATE
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

DATA_DIR = Path.cwd().parent / 'data' / 'raw' if Path.cwd().name == 'notebooks' else Path('data/raw')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'
print(f'Using data directory: {DATA_DIR.resolve()}')

Using data directory: C:\Users\niran\Documents\GitHub\AI-Commerce-Analytics-Platform\data\raw


## Dataset Loading

The loader applies nullable, memory-efficient types to dimensions and measures, while parsing event timestamps at ingestion. Categorical conversion is deferred until after loading so cardinality can be assessed safely.

In [2]:
TABLE_CONFIG = {
    'customers': {'file': 'customers.csv', 'dates': [], 'dtypes': {'customer_age': 'Int16', 'customer_zip_code_prefix': 'string'}},
    'geolocation': {'file': 'geolocation.csv', 'dates': [], 'dtypes': {'zip_code_prefix': 'string', 'geolocation_lat': 'float32', 'geolocation_lng': 'float32'}},
    'orders': {'file': 'orders.csv', 'dates': ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'], 'dtypes': {}},
    'order_items': {'file': 'order_items.csv', 'dates': ['shipping_limit_date'], 'dtypes': {'order_item_id': 'Int16', 'price': 'float32', 'freight_value': 'float32', 'discount_rate': 'float32'}},
    'order_payments': {'file': 'order_payments.csv', 'dates': [], 'dtypes': {'payment_sequential': 'Int16', 'payment_installments': 'Int16', 'payment_value': 'float32'}},
    'order_reviews': {'file': 'order_reviews.csv', 'dates': ['review_creation_date', 'review_answer_timestamp'], 'dtypes': {'review_score': 'Int8'}},
    'products': {'file': 'products.csv', 'dates': [], 'dtypes': {'product_weight_g': 'Int32', 'product_length_cm': 'Int16', 'product_height_cm': 'Int16', 'product_width_cm': 'Int16', 'cost': 'float32', 'price': 'float32'}},
    'sellers': {'file': 'sellers.csv', 'dates': [], 'dtypes': {'seller_contact_age': 'Int16', 'seller_zip_code_prefix': 'string'}},
}

def read_table(name: str, config: dict) -> pd.DataFrame:
    """Read one source table with explicit parsing and clear failures."""
    path = DATA_DIR / config['file']
    if not path.exists():
        raise FileNotFoundError(f'Missing required source: {path}')
    return pd.read_csv(path, dtype=config['dtypes'], parse_dates=config['dates'], low_memory=False)

tables = {name: read_table(name, config) for name, config in TABLE_CONFIG.items()}
globals().update(tables)

load_summary = pd.DataFrame([
    {'table': name, 'rows': len(frame), 'columns': frame.shape[1], 'memory_mb': frame.memory_usage(deep=True).sum() / 1e6}
    for name, frame in tables.items()
]).sort_values('memory_mb', ascending=False)
display(load_summary.style.format({'rows': '{:,.0f}', 'memory_mb': '{:,.1f}'}))

,table,rows,columns,memory_mb
3,order_items,"2,199,819",8,611.5
0,customers,"1,000,000",9,504.3
5,order_reviews,"933,748",7,313.8
2,orders,"1,000,000",8,267.9
4,order_payments,"1,149,371",5,176.5
1,geolocation,"11,500",5,2.0
6,products,"2,000",10,0.6
7,sellers,500,8,0.2


## Schema Validation and Data Dictionary

The expected-column contract below catches unexpected upstream schema drift. Types are then profiled from the actual loaded data. This is a validation aid, not a substitute for a versioned production data contract.

In [3]:
EXPECTED_COLUMNS = {
    'customers': {'customer_id', 'customer_unique_id', 'customer_name', 'customer_gender', 'customer_age', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_segment'},
    'geolocation': {'zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state'},
    'orders': {'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'},
    'order_items': {'order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'discount_rate'},
    'order_payments': {'order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value'},
    'order_reviews': {'review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp'},
    'products': {'product_id', 'product_category_name', 'product_name', 'product_brand', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'cost', 'price'},
    'sellers': {'seller_id', 'seller_company_name', 'seller_contact_name', 'seller_contact_gender', 'seller_contact_age', 'seller_zip_code_prefix', 'seller_city', 'seller_state'},
}

schema_results = []
for name, expected in EXPECTED_COLUMNS.items():
    actual = set(tables[name].columns)
    schema_results.append({'table': name, 'status': 'PASS' if actual == expected else 'CHECK', 'missing_columns': ', '.join(sorted(expected - actual)) or '—', 'unexpected_columns': ', '.join(sorted(actual - expected)) or '—'})
display(pd.DataFrame(schema_results))

data_dictionary = pd.concat([
    pd.DataFrame({'table': name, 'column': frame.columns, 'dtype': frame.dtypes.astype(str).values, 'non_null': frame.notna().sum().values, 'unique_values': frame.nunique(dropna=True).values})
    for name, frame in tables.items()
], ignore_index=True)
display(data_dictionary.sort_values(['table', 'column']).reset_index(drop=True))

,table,status,missing_columns,unexpected_columns
0,customers,PASS,—,—
1,geolocation,PASS,—,—
2,orders,PASS,—,—
3,order_items,PASS,—,—
4,order_payments,PASS,—,—
5,order_reviews,PASS,—,—
6,products,PASS,—,—
7,sellers,PASS,—,—


,table,column,dtype,non_null,unique_values
0,customers,customer_age,Int16,1000000,53
1,customers,customer_city,str,1000000,10
2,customers,customer_gender,str,1000000,2
3,customers,customer_id,str,1000000,1000000
4,customers,customer_name,str,1000000,152603
5,customers,customer_segment,str,1000000,3
6,customers,customer_state,str,1000000,10
7,customers,customer_unique_id,str,1000000,279199
8,customers,customer_zip_code_prefix,string,1000000,900
9,geolocation,geolocation_city,str,11500,10


## Data Quality Assessment

Quality checks measure missingness, exact-row duplication, and relational integrity. Null timestamps can be operationally legitimate (for example, cancellation before delivery), so they are reported rather than blindly imputed.

In [4]:
def missingness(frame: pd.DataFrame, table_name: str) -> pd.DataFrame:
    result = frame.isna().sum().rename('missing_count').to_frame()
    result['missing_pct'] = result['missing_count'].div(len(frame)).mul(100)
    result['table'] = table_name
    return result.reset_index(names='column')

missing_summary = pd.concat([missingness(frame, name) for name, frame in tables.items()], ignore_index=True)
missing_summary = missing_summary.query('missing_count > 0').sort_values('missing_pct', ascending=False)
display(missing_summary.style.format({'missing_count': '{:,.0f}', 'missing_pct': '{:.2f}%'}))

if not missing_summary.empty:
    fig = px.bar(missing_summary.head(20), x='missing_pct', y='column', color='table', orientation='h', title='Top 20 fields by missing-value rate', labels={'missing_pct': 'Missing values (%)'})
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
    fig.show()

duplicate_summary = pd.DataFrame([{'table': name, 'exact_duplicate_rows': int(frame.duplicated().sum()), 'duplicate_pct': frame.duplicated().mean() * 100} for name, frame in tables.items()])
display(duplicate_summary.style.format({'exact_duplicate_rows': '{:,.0f}', 'duplicate_pct': '{:.3f}%'}))

,column,missing_count,missing_pct,table
19,order_delivered_carrier_date,"66,252",6.63%,orders
20,order_delivered_customer_date,"66,252",6.63%,orders


,table,exact_duplicate_rows,duplicate_pct
0,customers,0,0.000%
1,geolocation,0,0.000%
2,orders,0,0.000%
3,order_items,0,0.000%
4,order_payments,0,0.000%
5,order_reviews,0,0.000%
6,products,0,0.000%
7,sellers,0,0.000%


In [5]:
## Referential Integrity & Primary/Foreign Key Validation

KEY_RULES = [
    ('customers', ['customer_id']), ('products', ['product_id']), ('sellers', ['seller_id']), ('orders', ['order_id']),
    ('order_items', ['order_id', 'order_item_id']), ('order_payments', ['order_id', 'payment_sequential']), ('order_reviews', ['review_id']),
]
FOREIGN_KEYS = [
    ('orders', 'customer_id', 'customers', 'customer_id'), ('order_items', 'order_id', 'orders', 'order_id'),
    ('order_items', 'product_id', 'products', 'product_id'), ('order_items', 'seller_id', 'sellers', 'seller_id'),
    ('order_payments', 'order_id', 'orders', 'order_id'), ('order_reviews', 'order_id', 'orders', 'order_id'),
]

pk_results = []
for table_name, keys in KEY_RULES:
    frame = tables[table_name]
    null_keys = int(frame[keys].isna().any(axis=1).sum())
    duplicate_keys = int(frame.duplicated(keys).sum())
    pk_results.append({'table': table_name, 'key': ' + '.join(keys), 'null_key_rows': null_keys, 'duplicate_key_rows': duplicate_keys, 'status': 'PASS' if not null_keys and not duplicate_keys else 'CHECK'})
pk_validation = pd.DataFrame(pk_results)
display(pk_validation)

fk_results = []
for child, child_key, parent, parent_key in FOREIGN_KEYS:
    child_values = tables[child][child_key].dropna()
    parent_values = set(tables[parent][parent_key].dropna())
    orphan_count = int((~child_values.isin(parent_values)).sum())
    fk_results.append({'relationship': f'{child}.{child_key} → {parent}.{parent_key}', 'child_rows': len(child_values), 'orphan_rows': orphan_count, 'orphan_pct': orphan_count / len(child_values) * 100 if len(child_values) else 0, 'status': 'PASS' if orphan_count == 0 else 'CHECK'})
fk_validation = pd.DataFrame(fk_results)
display(fk_validation.style.format({'child_rows': '{:,.0f}', 'orphan_rows': '{:,.0f}', 'orphan_pct': '{:.4f}%'}))

,table,key,null_key_rows,duplicate_key_rows,status
0,customers,customer_id,0,0,PASS
1,products,product_id,0,0,PASS
2,sellers,seller_id,0,0,PASS
3,orders,order_id,0,0,PASS
4,order_items,order_id + order_item_id,0,0,PASS
5,order_payments,order_id + payment_sequential,0,0,PASS
6,order_reviews,review_id,0,0,PASS


,relationship,child_rows,orphan_rows,orphan_pct,status
0,orders.customer_id → customers.customer_id,"1,000,000",0,0.0000%,PASS
1,order_items.order_id → orders.order_id,"2,199,819",0,0.0000%,PASS
2,order_items.product_id → products.product_id,"2,199,819",0,0.0000%,PASS
3,order_items.seller_id → sellers.seller_id,"2,199,819",0,0.0000%,PASS
4,order_payments.order_id → orders.order_id,"1,149,371",0,0.0000%,PASS
5,order_reviews.order_id → orders.order_id,"933,748",0,0.0000%,PASS


## Memory Optimization

Low-cardinality text columns are converted to `category` after profiling. The optimization is safe for this notebook because categories preserve values; production pipelines should persist and version their dtype contract.

In [6]:
def optimize_categories(frame: pd.DataFrame, threshold: float = 0.20) -> pd.DataFrame:
    """Convert low-cardinality object/string columns to categorical dtype."""
    optimized = frame.copy()
    for column in optimized.select_dtypes(include=['object', 'string']).columns:
        cardinality_ratio = optimized[column].nunique(dropna=False) / max(len(optimized), 1)
        if cardinality_ratio <= threshold:
            optimized[column] = optimized[column].astype('category')
    return optimized

before_mb = sum(frame.memory_usage(deep=True).sum() for frame in tables.values()) / 1e6
tables = {name: optimize_categories(frame) for name, frame in tables.items()}
globals().update(tables)
after_mb = sum(frame.memory_usage(deep=True).sum() for frame in tables.values()) / 1e6
print(f'Memory footprint: {before_mb:,.1f} MB → {after_mb:,.1f} MB ({(1 - after_mb / before_mb) * 100:,.1f}% reduction)')

Memory footprint: 1,876.9 MB → 939.6 MB (49.9% reduction)


## Customer, Product, and Seller Analysis

A reusable order-level revenue table avoids fan-out errors from directly joining orders to item and payment detail. Gross merchandise value (GMV) is item price plus freight, net of the recorded item discount rate. Payment value is retained separately because it can differ from GMV through payment-level adjustments or multi-payment orders.

In [7]:
order_items = order_items.assign(item_gmv=(order_items['price'] + order_items['freight_value']) * (1 - order_items['discount_rate'].fillna(0)))
order_value = order_items.groupby('order_id', observed=True).agg(items=('order_item_id', 'size'), gmv=('item_gmv', 'sum'), product_revenue=('price', 'sum'), freight_revenue=('freight_value', 'sum')).reset_index()
payment_value = order_payments.groupby('order_id', observed=True)['payment_value'].sum().rename('payment_value').reset_index()
order_fact = orders.merge(order_value, on='order_id', how='left', validate='one_to_one').merge(payment_value, on='order_id', how='left', validate='one_to_one')
order_fact['gmv'] = order_fact['gmv'].fillna(0)
order_fact['delivery_days'] = (order_fact['order_delivered_customer_date'] - order_fact['order_purchase_timestamp']).dt.total_seconds() / 86_400
order_fact['delivery_vs_estimate_days'] = (order_fact['order_delivered_customer_date'] - order_fact['order_estimated_delivery_date']).dt.total_seconds() / 86_400

customer_metrics = order_fact.merge(customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_segment']], on='customer_id', how='left', validate='many_to_one').groupby(['customer_unique_id', 'customer_state', 'customer_segment'], observed=True).agg(orders=('order_id', 'nunique'), gmv=('gmv', 'sum'), avg_order_value=('gmv', 'mean')).reset_index()
print(f'Unique customers: {customer_metrics.customer_unique_id.nunique():,}; repeat-purchase rate: {(customer_metrics.orders.gt(1).mean() * 100):.2f}%')

category_metrics = order_items.merge(products[['product_id', 'product_category_name', 'cost']], on='product_id', how='left', validate='many_to_one').assign(gross_profit=lambda x: (x['price'] - x['cost']) * (1 - x['discount_rate'].fillna(0))).groupby('product_category_name', observed=True).agg(items=('order_item_id', 'size'), gmv=('item_gmv', 'sum'), gross_profit=('gross_profit', 'sum')).reset_index().sort_values('gmv', ascending=False)
fig = px.bar(category_metrics.head(12), x='product_category_name', y='gmv', hover_data=['items', 'gross_profit'], title='Top product categories by GMV', labels={'product_category_name': 'Category', 'gmv': 'GMV'})
fig.update_xaxes(tickangle=-35)
fig.show()

seller_metrics = order_items.groupby('seller_id', observed=True).agg(items=('order_item_id', 'size'), gmv=('item_gmv', 'sum'), orders=('order_id', 'nunique')).reset_index().merge(sellers[['seller_id', 'seller_company_name', 'seller_state']], on='seller_id', how='left', validate='one_to_one').sort_values('gmv', ascending=False)
fig = px.bar(seller_metrics.head(15), x='seller_company_name', y='gmv', hover_data=['orders', 'items', 'seller_state'], title='Top 15 sellers by GMV', labels={'seller_company_name': 'Seller', 'gmv': 'GMV'})
fig.update_xaxes(tickangle=-35)
fig.show()
display(Markdown('**Business interpretation:** Concentration in a small set of categories or sellers creates commercial leverage, but also supplier and assortment risk. Use the displayed rankings to prioritize margin reviews, seller SLAs, and catalog investment.'))

Unique customers: 279,199; repeat-purchase rate: 84.06%


**Business interpretation:** Concentration in a small set of categories or sellers creates commercial leverage, but also supplier and assortment risk. Use the displayed rankings to prioritize margin reviews, seller SLAs, and catalog investment.

## Orders, Payments, and Reviews Analysis

Order status and purchase trends describe demand and operational throughput. Delivery delay is calculated only where both delivered and estimated dates exist; a positive value means delivery occurred after the promise date.

In [8]:
status_summary = order_fact.groupby('order_status', observed=True).agg(orders=('order_id', 'size'), gmv=('gmv', 'sum')).reset_index().sort_values('orders', ascending=False)
fig = px.bar(status_summary, x='order_status', y='orders', color='gmv', title='Order volume and GMV by status', labels={'order_status': 'Order status', 'orders': 'Orders'})
fig.show()

monthly_orders = order_fact.dropna(subset=['order_purchase_timestamp']).assign(month=lambda x: x['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()).groupby('month', observed=True).agg(orders=('order_id', 'size'), gmv=('gmv', 'sum')).reset_index()
fig = px.line(monthly_orders, x='month', y='gmv', markers=True, title='Monthly GMV trend', labels={'month': 'Purchase month', 'gmv': 'GMV'})
fig.show()

payment_summary = order_payments.groupby('payment_type', observed=True).agg(payments=('order_id', 'size'), payment_value=('payment_value', 'sum'), avg_installments=('payment_installments', 'mean')).reset_index().sort_values('payment_value', ascending=False)
display(payment_summary.style.format({'payments': '{:,.0f}', 'payment_value': '{:,.2f}', 'avg_installments': '{:.2f}'}))
fig = px.pie(payment_summary, names='payment_type', values='payment_value', title='Payment value mix')
fig.show()

review_summary = order_reviews.groupby('review_score', observed=True).size().rename('reviews').reset_index()
fig = px.bar(review_summary, x='review_score', y='reviews', title='Review-score distribution', labels={'review_score': 'Review score', 'reviews': 'Reviews'})
fig.show()

reviews_with_delivery = order_reviews[['order_id', 'review_score']].merge(order_fact[['order_id', 'delivery_days', 'delivery_vs_estimate_days']], on='order_id', how='inner', validate='many_to_one').dropna(subset=['delivery_days'])
delivery_review = reviews_with_delivery.groupby('review_score', observed=True).agg(orders=('order_id', 'size'), median_delivery_days=('delivery_days', 'median'), late_delivery_rate=('delivery_vs_estimate_days', lambda x: (x > 0).mean() * 100)).reset_index()
display(delivery_review.style.format({'orders': '{:,.0f}', 'median_delivery_days': '{:.2f}', 'late_delivery_rate': '{:.2f}%'}))
display(Markdown('**Business interpretation:** Compare the delivery and lateness metrics by score before inferring causality. A persistent association would motivate service-recovery experiments and delivery-risk features for churn or sentiment models.'))

,payment_type,payments,payment_value,avg_installments
3,credit_card,"536,416","620,220,672.00",6.41
5,paypal,"181,278","193,706,864.00",1.00
0,apple_pay,"92,114","118,802,888.00",1.00
1,bank_transfer,"96,766","94,242,928.00",1.00
6,voucher,"123,150","91,807,304.00",1.00
4,debit_card,"91,034","88,510,936.00",6.34
2,boleto,"28,613","14,892,989.00",1.00


,review_score,orders,median_delivery_days,late_delivery_rate
0,1,"45,808",17.04,100.00%
1,2,"115,300",8.35,40.10%
2,3,"68,823",6.38,0.00%
3,4,"234,499",6.39,0.00%
4,5,"469,318",6.39,0.00%


**Business interpretation:** Compare the delivery and lateness metrics by score before inferring causality. A persistent association would motivate service-recovery experiments and delivery-risk features for churn or sentiment models.

## Geographic Analysis, Correlation Analysis, and Executive KPIs

In [9]:
state_metrics = order_fact.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left', validate='many_to_one').groupby('customer_state', observed=True).agg(orders=('order_id', 'size'), gmv=('gmv', 'sum'), avg_delivery_days=('delivery_days', 'mean')).reset_index().sort_values('gmv', ascending=False)
fig = px.bar(state_metrics.head(20), x='customer_state', y='gmv', color='avg_delivery_days', hover_data=['orders'], title='Top customer states: GMV and average delivery time', labels={'customer_state': 'State', 'gmv': 'GMV'})
fig.show()

numeric_columns = ['gmv', 'items', 'product_revenue', 'freight_revenue', 'payment_value', 'delivery_days', 'delivery_vs_estimate_days']
correlation = order_fact[numeric_columns].corr(numeric_only=True)
fig = px.imshow(correlation, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Order-level feature correlations')
fig.show()

delivered = order_fact['order_status'].eq('delivered')
kpis = {
    'Orders': f"{len(order_fact):,}",
    'GMV': f"${order_fact['gmv'].sum():,.0f}",
    'Average order value': f"${order_fact.loc[order_fact['gmv'] > 0, 'gmv'].mean():,.2f}",
    'Delivery rate': f"{delivered.mean() * 100:.2f}%",
    'Median delivery days': f"{order_fact.loc[delivered, 'delivery_days'].median():.1f}",
    'Average review score': f"{order_reviews['review_score'].mean():.2f} / 5",
}
display(pd.DataFrame(kpis.items(), columns=['KPI', 'Value']))
display(Markdown('**Executive interpretation:** The KPI table is the refreshable baseline for the future executive dashboard. Treat trend changes as signals to investigate by cohort, seller, category, state, and fulfilment stage—not as standalone explanations.'))

,KPI,Value
0,Orders,"1,000,000"
1,GMV,"$1,213,510,016"
2,Average order value,"$1,213.51"
3,Delivery rate,93.37%
4,Median delivery days,7.1
5,Average review score,4.03 / 5


**Executive interpretation:** The KPI table is the refreshable baseline for the future executive dashboard. Treat trend changes as signals to investigate by cohort, seller, category, state, and fulfilment stage—not as standalone explanations.

## Business Insights

Use the computed outputs above as the source of truth. The following opportunity themes should be prioritized when their corresponding metrics show material concentration or deterioration:

- **Retention:** repeat-purchase distribution, customer segment performance, and post-delivery satisfaction support lifecycle programs.
- **Fulfilment:** late-delivery rates and state/seller differences identify SLA, inventory-placement, and carrier-management opportunities.
- **Commercial mix:** category GMV and gross profit distinguish demand drivers from margin drivers.
- **Payment design:** payment mix and instalment behavior inform checkout optimization and financing strategy.
- **Experience recovery:** review scores joined to delivery features help target service interventions.

## Feature Engineering Opportunities

| Domain | Candidate features | Downstream use |
|---|---|---|
| Customer | recency, frequency, monetary value, segment, state, repeat rate | CLV, churn, segmentation |
| Order | basket size, GMV, discount, freight share, purchase month/day | forecasting, churn, dashboard |
| Delivery | approval lag, carrier lag, delivery days, promised-date variance | delay prediction, service recovery |
| Product/seller | category, margin, seller history, dimensions/weight | recommendations, delivery prediction |
| Review | score, title/message presence, response lag, text embeddings | sentiment analysis, churn |

## Machine Learning Readiness Checklist

- [ ] Resolve every `CHECK` finding from the schema, key, and foreign-key validation tables.
- [ ] Define event-time cutoffs and prevent future information from entering training features.
- [ ] Set task-specific labels and prediction horizons with business owners.
- [ ] Split data chronologically; retain a final untouched test period.
- [ ] Fit imputers, encoders, scalers, and text vectorizers on training data only.
- [ ] Assess class balance, subgroup performance, and potential geographic/segment bias.
- [ ] Version raw data, feature definitions, model inputs, and validation results.

## Production Considerations

Implement source-level data contracts, row-count and freshness monitors, key-integrity tests, and alert thresholds for null-rate or distribution drift. For large refreshes, move CSV ingestion to partitioned Parquet with an orchestrated warehouse/lakehouse pipeline. Aggregate before joining one-to-many tables, enforce join validation, and publish a documented semantic layer for GMV, revenue, and delivery KPIs.

## Key Takeaways

This notebook establishes the validated analytical grain: orders are the central fact, item and payment detail are aggregated before integration, and customer/product/seller attributes provide dimensions. The quality outputs determine whether downstream notebooks can proceed without remediation; the KPI, delivery, and satisfaction analyses identify the highest-value hypotheses to test next.